# Conv1D (time-wise) + BiLSTM (Experiment)

**Challenge:** AN2DL 2024 — Pirate Pain Classification  
**Approach:** 1D convolution applied along the **time axis** (collapsing the window dimension) as a  
feature aggregator before a Bidirectional LSTM. Categorical features encoded via `nn.Embedding`.  
**Result:** val F1 ≈ 0.93 — close to the best model but slightly lower; the time-wise conv  
compresses temporal context before the LSTM rather than extracting cross-feature patterns.

### Pipeline
1. Drop zero-variance `joint_30`
2. Separate features: embeddings (`n_legs`), joint angles (30), pain surveys (4)
3. Stratified train/val split (60 val samples)
4. Min-max normalisation on joint columns (train statistics only)
5. Sliding-window sequences (window=20, stride=5)
6. `nn.Embedding(vocab=2, dim=3)` for `n_legs`
7. `Conv1D(in=WINDOW_SIZE, out=1, kernel=2)` on joints → BiLSTM(128) → Linear(3)
8. Weighted `CrossEntropyLoss`, AdamW, early stopping (patience=50)

In [ ]:
#Libraries import

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # Plotting 
import seaborn as sns # Statistical data visualization
import warnings # To suppress warnings
import torch 
from torch import nn #Neural Networks 
from torch.utils.data import TensorDataset, DataLoader
import random
import logging   
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix  # Usefull metrics
from sklearn.model_selection import train_test_split
from scipy import stats
import re           # Regex

In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
SEED = 42 # To guarantee reproducibility
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Set torch options

logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models


# Use a GPU if available, or use a CPU instead (N.B. GPUs are made available from kaggle)
if torch.cuda.is_available():     
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

# Check torch version 
print(f"PyTorch version: {torch.__version__}")


# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

In [ ]:
#Set environment variables

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

In [ ]:
#Suppress warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

In [ ]:
#Data Loading

X = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train.csv')
y = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train_labels.csv')
X_test = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_test.csv')

In [ ]:
X.drop('joint_30', axis=1, inplace=True)
X_test.drop('joint_30', axis=1, inplace=True)

In [ ]:
# How many samples?
samples = X['sample_index'].unique()
n_samples = len(samples) 
print(n_samples) # 661

In [ ]:
label_map = {
    'no_pain' : 0,
    'low_pain' : 1,
    'high_pain' : 2,
    0 : 0,
    1 : 1,
    2 : 2
}

y['label'] = y['label'].map(label_map)

In [ ]:
sample_labels = []

for sample in samples:
    sample_labels.append([sample, int(y[y['sample_index']==sample]['label'].unique())])

In [ ]:
labels_df = pd.DataFrame(sample_labels)
print(labels_df)

In [ ]:
N_VAL_SAMPLES = 60 # Number of samples used for validation (tunable parameter)

val_perc =  N_VAL_SAMPLES/len(samples)

n_train_samples = len(samples) - N_VAL_SAMPLES

print(n_train_samples)
print(val_perc)

In [ ]:
train_samples, val_samples = train_test_split(labels_df, test_size = N_VAL_SAMPLES, random_state = SEED, shuffle=True, stratify=labels_df[1])

In [ ]:
train_samples = train_samples[0]
val_samples = val_samples[0]

In [ ]:
print(len(train_samples))

In [ ]:
print(len(val_samples))

In [ ]:
emb_cols = ['sample_index', 'time', 'n_legs']  # n_hands and n_eyes provide the same informations, se we discard them

joint_cols = [col for col in X.columns if re.match(r'joint_\d{2}', col)]
joint_cols.append('sample_index')
joint_cols.append('time')

pain_cols = [col for col in X.columns if re.match(r'pain_survey.*', col)]
pain_cols.append('sample_index')
pain_cols.append('time')

In [ ]:
X_emb = X[emb_cols]
X_test_emb = X_test[emb_cols]

In [ ]:
X_joint = X[joint_cols]
X_test_joint = X_test[joint_cols]

In [ ]:
X_pain = X[pain_cols]
X_test_pain = X_test[pain_cols]

In [ ]:
# Split the dataset into training and validation sets based on samples ids
X_train_emb = X_emb[X_emb['sample_index'].isin(train_samples)]
X_val_emb = X_emb[X_emb['sample_index'].isin(val_samples)]
X_train_joint = X_joint[X_joint['sample_index'].isin(train_samples)]
X_val_joint = X_joint[X_joint['sample_index'].isin(val_samples)]
X_train_pain = X_pain[X_pain['sample_index'].isin(train_samples)]
X_val_pain = X_pain[X_pain['sample_index'].isin(val_samples)]

In [ ]:
print(X_train_emb.shape)
print(X_train_joint.shape)
print(X_train_pain.shape)

In [ ]:
print(X_val_emb.shape)
print(X_val_joint.shape)
print(X_val_pain.shape)

In [ ]:
y_train = y[y['sample_index'].isin(train_samples)]
y_val = y[y['sample_index'].isin(val_samples)]

In [ ]:
print(len(y_train))
print(len(y_val))

In [ ]:
# Study labels distribution in training set 

training_labels = {
    0 : 0,
    1 : 0,
    2 : 0
}


# Count occurrences of each activity for unique IDs in the training set
for id in y_train['sample_index'].unique():
    label = y_train[y_train['sample_index'] == id]['label'].values[0]
    training_labels[label] += 1

# Print the distribution of training labels
print('Training labels:', training_labels)
print('Training distributions:\n')
for label in training_labels:
    stat = training_labels[label]/ (training_labels[0] + training_labels[1] + training_labels[2])
    print(f'{label}: {stat}')

In [ ]:
# Study labels distribution in validation set 

validation_labels = {
    0 : 0,
    1 : 0,
    2 : 0
}


# Count occurrences of each activity for unique IDs in the validation set
for id in y_val['sample_index'].unique():
    label = y_val[y_val['sample_index'] == id]['label'].values[0]
    validation_labels[label] += 1


# Print the distribution of validation labels
print('Validation labels:', validation_labels)
print('Validation distributions:\n')
for label in validation_labels:
    stat = validation_labels[label]/ (validation_labels[0] + validation_labels[1] + validation_labels[2])
    print(f'{label}: {stat}')

In [ ]:
X_train_emb['n_legs'] = X_train_emb['n_legs'].replace('two', 'two_legs')
print(X_train_emb.head())

X_val_emb['n_legs'] = X_val_emb['n_legs'].replace('two', 'two_legs')
print(X_val_emb.head())

X_test_emb['n_legs'] = X_test_emb['n_legs'].replace('two', 'two_legs')
print(X_test_emb.head())

In [ ]:
embedding_map = {
    'two_legs' : 0,
    'one+peg_leg' : 1, 
}

In [ ]:
X_train_emb['n_legs'] = X_train_emb['n_legs'].map(embedding_map)
print(X_train_emb.head())


In [ ]:
X_val_emb['n_legs'] = X_val_emb['n_legs'].map(embedding_map)

print(X_val_emb.head())

In [ ]:
X_test_emb['n_legs'] = X_test_emb['n_legs'].map(embedding_map)


print(X_test_emb.head())

In [ ]:
joint_cols.remove('sample_index')
joint_cols.remove('time')

In [ ]:
mins = X_train_joint[joint_cols].min()
maxs = X_train_joint[joint_cols].max()

for col in joint_cols:
    den = maxs[col] - mins[col]
    
    if den == 0:
        den = maxs[col]
        
    X_train_joint[col] = (X_train_joint[col] - mins[col])/den # Normalize train set
    X_val_joint[col] = (X_val_joint[col] - mins[col])/den # Normalize val set
    X_test_joint[col] = (X_test_joint[col] - mins[col])/den # Normalize test set

In [ ]:
print(X_train_joint.head)

In [ ]:
print(X_train_joint.shape)

In [ ]:
training_labels = {
    0 : 0,
    1 : 0,
    2 : 0,
}


# Count occurrences of each activity for unique IDs in the training set
for id in y_train['sample_index'].unique():
    label = y_train[y_train['sample_index'] == id]['label'].values[0]
    training_labels[label] += 1

class_counts = torch.tensor([training_labels[0], training_labels[1], training_labels[2]], dtype=torch.float)

In [ ]:
print(class_counts)

In [ ]:
emb_cols.remove('sample_index')
emb_cols.remove('time')

pain_cols.remove('sample_index')
pain_cols.remove('time')

In [ ]:
print(emb_cols)
print(joint_cols)
print(pain_cols)

In [ ]:
y['label'] = y['label'].map(label_map)

In [ ]:
# Label mapping applied; y now has integer labels (0=no_pain, 1=low_pain, 2=high_pain)


In [ ]:
 # Window size and stride for sequence building

WINDOW_SIZE =20
STRIDE = 5 # the window need to be divisible by the stride

In [ ]:
# Function to build sequences

def build_sequences(df, window, stride, isTest, data_cols):
    # Be sure that the window is divisible by the stride
    assert window%stride == 0

    #Lists to store sequences and corresponding labels
    dataset = []
    labels = []

    # Iterate over the ids
    for id in df['sample_index'].unique():
        id_int = int(id)
        # Extract data from the current id
        temp = df[df['sample_index'] == id_int][data_cols].values

        if(not isTest):
            # Retrive the label for the id
            label = y[y['sample_index'] == id_int]['label'].values[0]

        # Calculate padding length to ensure full windows
        #padding_len = window - len(temp) % window

        # Create zero padding and concatenate with the data
        #padding = np.zeros((padding_len, len(data_cols)), dtype='float32')
        #temp = np.concatenate((temp, padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            if(not isTest):
                labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    if(not isTest):
        labels = np.array(labels)

    if(not isTest):
        return dataset, labels
    else:
        return dataset


In [ ]:
X_train_seq_emb, y_train_seq_emb = build_sequences(X_train_emb, WINDOW_SIZE, STRIDE, False, emb_cols)
X_val_seq_emb, y_val_seq_emb = build_sequences(X_val_emb, WINDOW_SIZE, STRIDE, False, emb_cols)
X_test_seq_emb = build_sequences(X_test_emb, WINDOW_SIZE, STRIDE, True, emb_cols)

In [ ]:
# Define the number of classes based on the categorical labels
num_classes = len(np.unique(y_train_seq_emb))
print(num_classes)

In [ ]:
X_train_seq_joint, y_train_seq_joint = build_sequences(X_train_joint, WINDOW_SIZE, STRIDE, False, joint_cols)
X_val_seq_joint, y_val_seq_joint = build_sequences(X_val_joint, WINDOW_SIZE, STRIDE, False, joint_cols)
X_test_seq_joint = build_sequences(X_test_joint, WINDOW_SIZE, STRIDE, True, joint_cols)

In [ ]:
X_train_seq_pain, y_train_seq_pain = build_sequences(X_train_pain, WINDOW_SIZE, STRIDE, False, pain_cols)
X_val_seq_pain, y_val_seq_pain = build_sequences(X_val_pain, WINDOW_SIZE, STRIDE, False, pain_cols)
X_test_seq_pain = build_sequences(X_test_pain, WINDOW_SIZE, STRIDE, True, pain_cols)

In [ ]:
print(X_val_seq_emb.shape)
print(X_val_seq_joint.shape)
print(X_val_seq_pain.shape)

In [ ]:
print(X_val_joint)

In [ ]:
train_ds_cat = TensorDataset(torch.from_numpy(X_train_seq_emb), torch.from_numpy(X_train_seq_joint), torch.from_numpy(X_train_seq_pain) ,torch.from_numpy(y_train_seq_emb))
val_ds_cat   = TensorDataset(torch.from_numpy(X_val_seq_emb), torch.from_numpy(X_val_seq_joint), torch.from_numpy(X_val_seq_pain) ,torch.from_numpy(y_val_seq_emb))
test_ds_cat = TensorDataset(torch.from_numpy(X_test_seq_emb), torch.from_numpy(X_test_seq_joint), torch.from_numpy(X_test_seq_pain))

In [ ]:
print(train_ds_cat.tensors[0].shape)
print(train_ds_cat.tensors[1].shape)
print(train_ds_cat.tensors[2].shape)
print(train_ds_cat.tensors[3].shape)

In [ ]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [ ]:
BATCH_SIZE = 512

In [ ]:
train_cat_loader = make_loader(train_ds_cat, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_cat_loader   = make_loader(val_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_cat_loader   = make_loader(test_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
# Training configuration
LEARNING_RATE = 1e-3
EPOCHS = 512
PATIENCE = 50

# Architecture
HIDDEN_LAYERS =  2       # Hidden layers
HIDDEN_SIZE = 128        # Neurons per layer

# Regularisation
DROPOUT_RATE = 0.2         # Dropout probability
L1_LAMBDA = 0            # L1 penalty
L2_LAMBDA = 0        # L2 penalty

In [ ]:
weights = 1/class_counts  # To give more importance to misclassifications of rare classes
weights = weights/weights.mean() # weights normalizations
print(weights)
weights = weights.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
class RecurrentClassifierConv1D(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_layers,
        num_classes,
        rnn_type,
        bidirectional=True,
        dropout_rate=0.2,
        vocab=2,
        embedding_dim=3,
        numerical_feat=30,
        embedded_feat=1,
        pain_feat=4
    ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        input_size = int(numerical_feat + embedded_feat * embedding_dim + pain_feat)
        self.embedding = nn.Embedding(vocab, embedding_dim)

        rnn_map = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]
        dropout_val = dropout_rate if num_layers > 1 else 0

        self.rnn = rnn_module(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        classifier_input_size = hidden_size * 2 if self.bidirectional else hidden_size
        self.classifier = nn.Linear(classifier_input_size, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x_emb, x_joint, x_pain):
        x_post_emb = self.embedding(x_emb.long())
        x_post_emb = x_post_emb.view(x_post_emb.size(0), x_post_emb.size(1), -1)

        x = torch.cat((x_joint, x_post_emb, x_pain), dim=-1)

        rnn_out, hidden = self.rnn(x)

        if self.rnn_type == 'LSTM':
            hidden = hidden[0]

        if self.bidirectional:
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            hidden_to_classify = hidden[-1]

        logits = self.classifier(hidden_to_classify)
        logits = self.log_softmax(logits)
        return logits


In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
  
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs_emb, inputs_joint, inputs_pain, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs_emb, inputs_joint, inputs_pain, targets = inputs_emb.to(device).float(), inputs_joint.to(device).float(), inputs_pain.to(device).float(), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs_emb, inputs_joint, inputs_pain)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs_emb.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [ ]:
def validate_one_epoch(model, val_loader, criterion, device):

    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs_emb, inputs_joint, inputs_pain, targets in val_loader:
            # Move data to device
            inputs_emb, inputs_joint, inputs_pain, targets = inputs_emb.to(device).float(), inputs_joint.to(device).float(), inputs_pain.to(device).float(), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs_emb, inputs_joint, inputs_pain)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs_emb.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [ ]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max', restore_best_weights=True, writer=None, verbose=10, experiment_name=""):
    
    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }
    
    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0
    
    print(f"Training {epochs} epochs...")
    
    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):
    
        # Forward pass through training data, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
    
        # Evaluate model on validation data without updating weights
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )
    
        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)
    
    
        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")
    
        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)
    
            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break
    
    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")
    
    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
    
    return model, training_history

In [ ]:
# Create model and display architecture with parameter count
lstm_conv = RecurrentClassifierConv1D(
    hidden_size=HIDDEN_SIZE,
    num_layers=HIDDEN_LAYERS,
    num_classes=num_classes,
    dropout_rate=DROPOUT_RATE,
    rnn_type='LSTM',
    ).to(device)


# Define optimizer with L2 regularization
optimizer = torch.optim.AdamW(lstm_conv.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)

# Enable mixed precision training for GPU acceleration
scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [ ]:
%%time

model_path = '/kaggle/working/models/lstm_con.pth'


# Train model and track training history
lstm_conv, training_history = fit(
    model=lstm_conv,
    train_loader=train_cat_loader,
    val_loader=val_cat_loader,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    verbose=1,
    experiment_name="lstm_emb",
    patience=PATIENCE
    )
    
# Update best model if current performance is superior
if training_history['val_f1'][-1] > best_performance:
    best_model = lstm_conv
    best_performance = training_history['val_f1'][-1]

# Save model
torch.save(lstm_conv.state_dict(), model_path)


In [ ]:
# Create a figure with two side-by-side subplots (two columns)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(training_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(training_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation accuracy on the second axis
ax2.plot(training_history['train_f1'], label='Training f1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(training_history['val_f1'], label='Validation f1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

In [ ]:
# Collect predictions and ground truth labels
val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb_emb, xb_joint, xb_pain, yb in val_cat_loader:
        xb_emb = xb_emb.to(device).float()
        xb_joint = xb_joint.to(device).float()
        xb_pain = xb_pain.to(device).float()

        # Forward pass: get model predictions
        logits = lstm_conv(xb_emb, xb_joint, xb_pain)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# Calculate overall validation metrics
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Visualise confusion matrix
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

In [ ]:
# Set model to evaluation mode
lstm_conv.eval()

# Lists to store predictions
sample_indices = []
predictions = []
all_preds = []

# Mapping from class index to label name
idx_to_label = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}



# Generate predictions
with torch.no_grad():  # Disable gradient computation for inference

    sample_count=0

    
    for batch_idx, (batch_emb, batch_joint, batch_pain) in enumerate(test_cat_loader):

        # batch is a list
        if isinstance(batch_emb, list):
            xb_emb = batch_emb[0]  # Get the first element (the features)
        else:
            xb_emb = batch_emb

        if isinstance(batch_joint, list):
            xb_joint = batch_joint[0]  # Get the first element (the features)
        else:
            xb_joint = batch_joint

        if isinstance(batch_pain, list):
            xb_pain = batch_pain[0]  # Get the first element (the features)
        else:
            xb_pain = batch_pain
        
        xb_emb = xb_emb.to(device).float()
        xb_joint = xb_joint.to(device).float()
        xb_pain = xb_pain.to(device).float()

        
        #Get model predictions
        logits = lstm_conv(xb_emb, xb_joint, xb_pain)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Convert predictions to labels and store
        for i, pred_idx in enumerate(preds):
            all_preds.append(pred_idx)

            
            if(((batch_idx*BATCH_SIZE)+i) % (X_test_seq_emb.shape[0] / 1324)  == 0):    
                sample_indices.append(f"{sample_count:03d}")
                predictions.append(idx_to_label[pred_idx])   
                sample_count += 1


submission_df = pd.DataFrame({
    'sample_index': sample_indices,
    'label': predictions
})


In [ ]:
submission_df.to_csv('/kaggle/working/submissions/submission_conv_strat_trying2.csv', index=False)

In [ ]:
print("\nPrediction distribution:")
print(submission_df['label'].value_counts())